# OOHScout AI — Real Waco Billboard Market (Chapter 4 Only)

**Goal:** Show every real permitted billboard on IH-35 through McLennan County
on an interactive map, with proof and verification links, using ONLY the
techniques taught in Chapter 4 of GeoAI Essentials.

**Data verified real (2026-08-28):**
- 182 permitted commercial signs in McLennan County (114 on IH-35)
- Source: TxDOT Commercial Signs public database

**Techniques used** — every single one from Chapter 4:
| Technique | Chapter 4 origin |
|-----------|-----------------|
| `requests.get()` REST API download | Ch4 Step 34 (NYC NTA) |
| `ox.features_from_polygon()` | Ch4 Steps 29, 32, 34 |
| Cache pattern | Ch4 (every step) |
| `.to_crs()` for meter projections | Ch4 Steps 27, 29 |
| `folium.Map`, `GeoJson`, `CircleMarker` | Ch4 Step 34 |
| `folium.FeatureGroup(show=True/False)` | Ch4 Step 34 |
| `folium.LayerControl` | Ch4 Step 34 |
| `.value_counts()` on GeoDataFrame | Ch4 Step 34 |

**Techniques deliberately NOT used** (from later chapters — you haven't learned yet):
- `gpd.sjoin_nearest()` (spatial joins)
- `unary_union`, `linemerge`, `.interpolate()` (line ops)
- `.buffer()` on lines (exclusion zones)
- Weighted scoring / ranking

```
Map layers priority (top-right layer control):
  ✅ Real Billboards       — 114 red dots, ON by default (THE STAR)
  ✅ IH-35 Corridor Line   — pink highway line, ON by default
  ✅ POI Advertiser Demand — blue dots, ON by default
  ☐  AADT Traffic Stations — OFF by default (advanced view)
```


In [20]:
# ── SECTION 1: SETUP ────────────────────────────────────────────────────────
# Only Chapter 4 imports — no scipy, no scikit-learn, no networkx.

import numpy as np
import pandas as pd
import geopandas as gpd
import osmnx as ox
import folium
import requests
import matplotlib.pyplot as plt
from pathlib import Path
from shapely.geometry import box
import warnings
warnings.filterwarnings('ignore')

# Data directory — same rule as every OOHScout notebook
DATA_DIR = Path('data/new_study')
DATA_DIR.mkdir(parents=True, exist_ok=True)

# ── STUDY AREA: Waco corridor bounding box ──────────────────────────────────
# [West, South, East, North] in WGS84 degrees — same format as Ch4 BBOX
WACO_BBOX = [-97.25, 31.30, -96.95, 31.85]

# ── REAL TxDOT ENDPOINTS (verified via live API on 2026-08-28) ──────────────
# We use requests.get() to hit these — exactly like Ch4 Step 34 hit the NYC API.

TXDOT_BILLBOARDS_URL = (
    'https://services.arcgis.com/KTcxiTD9dsQw4r7Z/ArcGIS/rest/services/'
    'Commercial_Signs_Test/FeatureServer/0/query'
)

TXDOT_AADT_URL = (
    'https://services.arcgis.com/KTcxiTD9dsQw4r7Z/ArcGIS/rest/services/'
    'TxDOT_AADT_Annuals_(Public_View)/FeatureServer/0/query'
)

# ── COORDINATE SYSTEMS ──────────────────────────────────────────────────────
WGS84   = 'EPSG:4326'      # storage/display
TEX_CRS = 'EPSG:32614'     # UTM Zone 14N — Texas, meters (like Ch4 EPSG:27700 for UK)

# ── POI CATEGORIES (billboard-relevant advertisers) ─────────────────────────
# Same style as Ch4 Step 34 which grabbed amenity=True for Manhattan.
# Here we narrow to categories that actually pay for OOH advertising.
ADVERTISER_TAGS = {
    'amenity': ['fuel', 'restaurant', 'fast_food', 'hotel', 'bank'],
    'shop':    ['supermarket', 'car_repair'],
    'tourism': ['hotel', 'motel'],
}

print('Setup complete.')
print(f'Study area (Waco, TX): {WACO_BBOX}')
print(f'Data dir: {DATA_DIR.resolve()}')


Setup complete.
Study area (Waco, TX): [-97.25, 31.3, -96.95, 31.85]
Data dir: C:\Users\nguye\Documents\billboardAI\notebooks\data\new_study


---
## Helper: Download from TxDOT REST API

This helper uses `requests.get()` — the exact same technique Chapter 4 used
in Step 34 to download NYC neighborhood boundaries from Socrata. The only
new thing is the URL and the `where` clause for filtering.


In [21]:
# ── HELPER: Query a TxDOT ArcGIS REST endpoint ──────────────────────────────
# Ch4 Step 34 used: requests.get(url, headers, params, timeout).raise_for_status()
# We use the same pattern here — just with more query parameters.

def fetch_txdot(endpoint_url, where='1=1', cache_path=None):
    """Download all records matching `where` from a TxDOT ArcGIS FeatureServer."""

    # Cache pattern (identical to Ch4 Step 29 buildings cache pattern)
    if cache_path is not None and cache_path.exists():
        print(f'  Loaded from cache: {cache_path.name}')
        return gpd.read_file(cache_path)

    # Collect features across pages (ArcGIS caps at 2000/response)
    all_features = []
    offset = 0
    while True:
        params = {
            'where': where,
            'outFields': '*',
            'resultOffset': offset,
            'resultRecordCount': 2000,
            'f': 'geojson',
        }
        # requests.get(...) with headers + timeout — same as Ch4 Step 34
        response = requests.get(
            endpoint_url,
            params=params,
            headers={'User-Agent': 'oohscout-ai/1.0 (educational)'},
            timeout=30,
        )
        response.raise_for_status()
        payload = response.json()

        features = payload.get('features', [])
        all_features.extend(features)
        print(f'  Fetched {len(features)} records (total: {len(all_features)})')

        exceeded = payload.get('properties', {}).get('exceededTransferLimit', False)
        if not exceeded or len(features) < 2000:
            break
        offset += 2000

    if not all_features:
        return gpd.GeoDataFrame(columns=['geometry'], crs=WGS84)

    # Build GeoDataFrame (same pattern as Ch4 Step 34 GeoDataFrame.from_features)
    gdf = gpd.GeoDataFrame.from_features(all_features, crs=WGS84)

    if cache_path is not None:
        gdf.to_file(cache_path, driver='GeoJSON')
        print(f'  Cached to: {cache_path.name}')

    return gdf


print('Helper ready: fetch_txdot(endpoint_url, where, cache_path)')


Helper ready: fetch_txdot(endpoint_url, where, cache_path)


---
# SECTION A — Real Billboards (THE STAR OF THE SHOW)

## What this section does

Downloads every permitted commercial sign on IH-35 in McLennan County from
TxDOT's public database. These are the REAL competitors any new billboard
must legally space itself against.

**Source of truth:** TxDOT Commercial Signs Regulatory Program
https://www.txdot.gov/business/right-of-way/commercial-signs-regulatory-program.html

**Verification:** Every popup on the map will link back to TxDOT so any
operator can confirm the data is real.


In [22]:
# ── A.1: Download all McLennan County billboards from TxDOT ─────────────────
# where="CNTY='McLennan'" filters at the API level — no local filtering needed.

billboards_path = DATA_DIR / 'ch4_txdot_billboards_mclennan.geojson'

print('Downloading real McLennan County billboards from TxDOT...')
billboards = fetch_txdot(
    TXDOT_BILLBOARDS_URL,
    where="CNTY='McLennan'",
    cache_path=billboards_path,
)

print(f'\nReal billboards in McLennan County: {len(billboards)}')
print(f'\nColumns from TxDOT: {list(billboards.columns)}')


  Fetched 182 records (total: 182)
  Cached to: ch4_txdot_billboards_mclennan.geojson

Real billboards in McLennan County: 182

Columns from TxDOT: ['geometry', 'OBJECTID', 'RCRD_ID', 'OWNR', 'LICNS', 'HWY', 'CNTY', 'STAT', 'ADDR', 'TYPE', 'LAT', 'LON', 'ELECTRONIC']


In [23]:
# ── A.2: Filter to IH-35 only (our corridor) ─────────────────────────────────
# Filter by highway just like Ch4 filtered building/POI features.

ih35_bb = billboards[billboards['HWY'] == 'IH 35'].copy().reset_index(drop=True)
print(f'Real billboards on IH-35 in McLennan County: {len(ih35_bb)}')

# Add a "verification URL" column for each billboard — used in popups
# TxDOT public map viewer accepts lat/lng in URL fragments
def verify_url(row):
    lat = row.geometry.y
    lon = row.geometry.x
    return (
        f'https://www.arcgis.com/apps/mapviewer/index.html'
        f'?webmap=d34f3091e0384dbfa98b8b503eb55967&center={lon},{lat}&level=17'
    )

ih35_bb['verify_url'] = ih35_bb.apply(verify_url, axis=1)

print(f'\nSample billboard record (row 0):')
sample = ih35_bb.iloc[0]
print(f'  Permit ID: {sample["RCRD_ID"]}')
print(f'  Operator:  {sample["OWNR"]}')
print(f'  Highway:   {sample["HWY"]}')
print(f'  Status:    {sample["STAT"]}')
print(f'  Digital:   {sample["ELECTRONIC"]}')
print(f'  Verify:    {sample["verify_url"][:70]}...')


Real billboards on IH-35 in McLennan County: 114

Sample billboard record (row 0):
  Permit ID: PMT-HBA-20585
  Operator:  Lamar Advantage Outdoor Company, L.P.
  Highway:   IH 35
  Status:    Active
  Digital:   No
  Verify:    https://www.arcgis.com/apps/mapviewer/index.html?webmap=d34f3091e0384d...


In [24]:
# ── A.3: Real market intelligence — operators, digital, top players ─────────
# .value_counts() is straight from Ch4 Step 34 ("pois_man['amenity'].value_counts()")

print('=' * 70)
print('  WACO IH-35 BILLBOARD MARKET — REAL DATA')
print('=' * 70)

print('\nOperators active on IH-35 in McLennan County:')
op_counts = ih35_bb['OWNR'].value_counts()
print(op_counts.to_string())

print(f'\nTotal unique operators: {len(op_counts)}')
print(f'Market leader: {op_counts.index[0]} ({op_counts.iloc[0]} signs)')
print(f'Market share of leader: {op_counts.iloc[0]/len(ih35_bb)*100:.1f}%')

print('\nDigital vs Static breakdown:')
elec_counts = ih35_bb['ELECTRONIC'].value_counts()
print(elec_counts.to_string())

digital_share = (ih35_bb['ELECTRONIC'] == 'Yes').sum() / len(ih35_bb) * 100
print(f'\nDigital penetration: {digital_share:.1f}% ({(ih35_bb["ELECTRONIC"] == "Yes").sum()} digital vs {(ih35_bb["ELECTRONIC"] == "No").sum()} static)')
print(f'\n→ Product angle: digital conversion opportunity is ~{100 - digital_share:.0f}% of the market')


  WACO IH-35 BILLBOARD MARKET — REAL DATA

Operators active on IH-35 in McLennan County:
OWNR
Lamar Advantage Outdoor Company, L.P.    51
Earl Louis Haberman, Jr                  25
ACME Partnership, LP                     11
FMG North Texas, LLC                      5
Outdoor Signs Joint Venture               3
Danny Harris                              3
Daniel Fairly                             2
MH Outdoor Media, LLC                     2
Edgelink Properties LLC                   2
Whitakerworks, LLC                        1
Tooley Media, LP                          1
Knox Oil of Texas, Inc.                   1
SignAd, Ltd.                              1
Newton Loy Harris                         1
KEM Texas, Ltd                            1
Primary Media Group, Inc.                 1
Joe F Frerich                             1
Archie Johnson                            1
Lockridge Outdoor Advertising             1

Total unique operators: 19
Market leader: Lamar Advantage Outdoor Com

---
# SECTION B — Advertiser Demand (POIs)

Uses `ox.features_from_polygon()` — the exact same technique Chapter 4 used
to download Manhattan amenity POIs. We narrow to billboard-relevant categories:
fuel, food, hotels, banks, supermarkets, auto services.

These are the businesses that would actually pay for a new billboard on IH-35.


In [25]:
# ── B.1: Download billboard-relevant POIs from OpenStreetMap ────────────────

pois_path = DATA_DIR / 'ch4_osm_pois_waco.geojson'

if pois_path.exists():
    pois = gpd.read_file(pois_path)
    print(f'Loaded from cache: {pois_path.name} — {len(pois)} POIs')
else:
    print('Downloading POIs from OSM...')
    corridor_poly = box(*WACO_BBOX)

    all_pois = []
    for tag_key, tag_values in ADVERTISER_TAGS.items():
        try:
            raw = ox.features_from_polygon(corridor_poly, tags={tag_key: tag_values})
            pts = raw[raw.geometry.geom_type == 'Point'].copy()
            if tag_key in pts.columns:
                pts = pts[['geometry', tag_key]].rename(columns={tag_key: 'category'})
            else:
                pts = pts[['geometry']].copy()
                pts['category'] = tag_key
            all_pois.append(pts)
            print(f'  {tag_key}: {len(pts)} POIs')
        except Exception as e:
            print(f'  {tag_key}: skipped ({e})')

    pois = pd.concat(all_pois, ignore_index=True)
    pois = pois[pois.geometry.is_valid].reset_index(drop=True)
    pois.to_file(pois_path, driver='GeoJSON')
    print(f'Saved {len(pois)} POIs to {pois_path.name}')

print(f'\nAdvertiser POIs in Waco corridor: {len(pois)}')
print(f'Top categories:')
print(pois['category'].value_counts().head(8).to_string())


  amenity: 155 POIs
  shop: 27 POIs
  tourism: 4 POIs
Saved 186 POIs to ch4_osm_pois_waco.geojson

Advertiser POIs in Waco corridor: 186
Top categories:
category
fast_food      62
restaurant     54
fuel           34
car_repair     18
supermarket     9
bank            5
hotel           4


---
# SECTION C — IH-35 Highway Line (Context)

Downloads the IH-35 road geometry from OpenStreetMap using the same
`ox.features_from_polygon()` pattern Ch4 Step 29 used for Edinburgh
buildings. We use it purely for visual context on the map — the pink line
showing where the highway runs.


In [26]:
# ── C.1: Download IH-35 road line for visual context ────────────────────────

roads_path = DATA_DIR / 'ch4_osm_ih35_waco.geojson'

if roads_path.exists():
    ih35_roads = gpd.read_file(roads_path)
    print(f'Loaded from cache: {roads_path.name}')
else:
    print('Downloading IH-35 from OSM...')
    corridor_poly = box(*WACO_BBOX)
    raw = ox.features_from_polygon(
        corridor_poly,
        tags={'highway': ['motorway', 'motorway_link']}
    )
    # Filter to line geometries (same pattern as Ch4 Step 29)
    ih35_roads = raw[raw.geometry.geom_type.isin(['LineString', 'MultiLineString'])].copy()
    ih35_roads = ih35_roads.reset_index(drop=True)
    ih35_roads.to_file(roads_path, driver='GeoJSON')

# Filter to mainlane only (not ramps)
if 'highway' in ih35_roads.columns:
    ih35_mainlane = ih35_roads[ih35_roads['highway'] == 'motorway'].copy()
else:
    ih35_mainlane = ih35_roads

print(f'IH-35 road segments: {len(ih35_mainlane)} (mainlane only)')

# Report length using UTM projection (Ch4 taught this — see Edinburgh step)
if len(ih35_mainlane) > 0:
    ih35_tex = ih35_mainlane.to_crs(TEX_CRS)
    total_km = ih35_tex.geometry.length.sum() / 1000
    print(f'Total IH-35 length in study area: {total_km:.1f} km')


IH-35 road segments: 430 (mainlane only)
Total IH-35 length in study area: 202.1 km


---
# SECTION D — AADT Traffic Data (OPTIONAL, hidden by default)

Downloads TxDOT AADT traffic counts using the same `requests.get()` REST
pattern as Section A. This layer will be **hidden by default** on the map
because it visually competes with the billboards.

Advanced users who want to see traffic distribution can toggle it on via
the layer control panel in the top-right of the map.


In [27]:
# ── D.1: Download AADT stations (optional advanced layer) ───────────────────

aadt_path = DATA_DIR / 'ch4_txdot_aadt_mclennan.geojson'

print('Downloading real McLennan AADT stations from TxDOT (optional layer)...')
aadt = fetch_txdot(
    TXDOT_AADT_URL,
    where="CNTY_NM='McLennan'",
    cache_path=aadt_path,
)

# Convert AADT numeric fields (ArcGIS sometimes returns as strings)
for col in aadt.columns:
    if col.startswith('AADT_RPT'):
        aadt[col] = pd.to_numeric(aadt[col], errors='coerce')

# Latest AADT: prefer current, fall back through history
if 'AADT_RPT_QTY' in aadt.columns:
    aadt['latest_aadt'] = aadt['AADT_RPT_QTY']
    hist_cols = sorted([c for c in aadt.columns if c.startswith('AADT_RPT_HIST_')])
    for hc in hist_cols:
        aadt['latest_aadt'] = aadt['latest_aadt'].fillna(aadt[hc])
    aadt['latest_aadt'] = pd.to_numeric(aadt['latest_aadt'], errors='coerce')

# Keep only stations with real data
aadt_with_data = aadt[aadt['latest_aadt'].notna() & (aadt['latest_aadt'] > 0)].copy()

print(f'\nAADT stations in McLennan County: {len(aadt)} ({len(aadt_with_data)} with valid counts)')
if len(aadt_with_data) > 0:
    print(f'AADT range: {int(aadt_with_data["latest_aadt"].min()):,} to {int(aadt_with_data["latest_aadt"].max()):,}')
    print(f'\nTop 5 highest-traffic stations:')
    top5_cols = ['TRFC_STATN_ID', 'latest_aadt']
    if 'ON_ROAD' in aadt_with_data.columns:
        top5_cols.insert(1, 'ON_ROAD')
    print(aadt_with_data.nlargest(5, 'latest_aadt')[top5_cols].to_string(index=False))


  Fetched 465 records (total: 465)
  Cached to: ch4_txdot_aadt_mclennan.geojson

AADT stations in McLennan County: 465 (465 with valid counts)
AADT range: 54 to 137,451

Top 5 highest-traffic stations:
TRFC_STATN_ID ON_ROAD  latest_aadt
      161H123  IH0035       137451
      161H122  IH0035       121794
     161H3504  IH0035       109927
       161H25  IH0035       100036
       161S6A  IH0035        98916


---
# SECTION E — The Map (Chapter 4 Style)

Uses only Ch4 Step 34 techniques: `folium.Map`, `folium.CircleMarker`,
`folium.FeatureGroup(show=True/False)`, `folium.Popup`, `folium.LayerControl`.

**Layer priorities:**
1. Real billboards — **ON by default** (the star)
2. IH-35 highway line — **ON by default** (context)
3. POI advertiser demand — **ON by default** (opportunity signal)
4. AADT traffic stations — **OFF by default** (advanced)

Every billboard popup includes THREE ways to verify the data:
1. Direct TxDOT map viewer link (opens the official map with the pin selected)
2. TxDOT Commercial Signs program page link
3. The raw permit ID for manual search


In [28]:
# ── E.1: Build the Chapter-4-style map ───────────────────────────────────────

# Compute center from real billboard bounds (Ch4 style — no fit_bounds needed)
bounds = ih35_bb.total_bounds  # [minx, miny, maxx, maxy]
center_lat = (bounds[1] + bounds[3]) / 2
center_lon = (bounds[0] + bounds[2]) / 2

fmap = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=11,
    tiles='OpenStreetMap',
)

# ── LAYER 1 (top priority): IH-35 highway line ──────────────────────────────
# folium.GeoJson pattern from Ch4 Step 34 (NYC neighborhoods)
highway_layer = folium.FeatureGroup(name='IH-35 Highway (context)', show=True)
folium.GeoJson(
    ih35_mainlane.__geo_interface__,
    style_function=lambda x: {
        'color': '#CC0066', 'weight': 4, 'opacity': 0.8,
    },
).add_to(highway_layer)
highway_layer.add_to(fmap)

# ── LAYER 2 (THE STAR): Real permitted billboards ───────────────────────────
# folium.CircleMarker + folium.Popup pattern from Ch4 Step 34
billboards_layer = folium.FeatureGroup(
    name=f'Real TxDOT Billboards ({len(ih35_bb)} on IH-35)',
    show=True,
)

for _, row in ih35_bb.iterrows():
    is_digital = row.get('ELECTRONIC') == 'Yes'
    color = '#FF2222' if is_digital else '#8B0000'
    permit_id = row.get('RCRD_ID', 'N/A')
    operator  = row.get('OWNR', 'N/A')
    verify_url = row.get('verify_url', '#')

    # THREE verification links in every popup
    popup_html = f'''
    <div style="font-family: sans-serif; font-size: 13px; min-width: 260px;">
      <b style="font-size: 14px; color: #8B0000;">REAL TxDOT PERMIT</b><br>
      <hr style="margin: 6px 0;">
      <b>Permit ID:</b> <code>{permit_id}</code><br>
      <b>Operator:</b> {operator}<br>
      <b>Highway:</b> {row.get('HWY', '')}<br>
      <b>County:</b> {row.get('CNTY', '')}<br>
      <b>Status:</b> {row.get('STAT', '')}<br>
      <b>Type:</b> {'Digital Bulletin' if is_digital else 'Static Bulletin'}<br>
      <b>License #:</b> {row.get('LICNS', 'N/A')}<br>
      <hr style="margin: 6px 0;">
      <b>Verify this permit (3 ways):</b><br>
      <ol style="margin: 4px 0 4px 20px; padding: 0;">
        <li>
          <a href="{verify_url}" target="_blank">
            Open TxDOT map viewer at this location
          </a>
        </li>
        <li>
          <a href="https://www.txdot.gov/business/right-of-way/commercial-signs-regulatory-program.html"
             target="_blank">
            TxDOT Commercial Signs program page
          </a>
        </li>
        <li>
          Copy permit ID <code>{permit_id}</code> and search in TxDOT database
        </li>
      </ol>
    </div>
    '''

    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=8,
        color='black', weight=1,
        fill=True, fill_color=color, fill_opacity=0.9,
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=f'{operator} — {permit_id}',
    ).add_to(billboards_layer)

billboards_layer.add_to(fmap)

# ── LAYER 3: POI advertiser demand ──────────────────────────────────────────
poi_layer = folium.FeatureGroup(
    name=f'POI Advertiser Demand ({len(pois)} businesses)',
    show=True,
)
for _, row in pois.iterrows():
    if row.geometry.geom_type != 'Point':
        continue
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=3,
        color='#004488', weight=0,
        fill=True, fill_color='#0066CC', fill_opacity=0.6,
        popup=f'<b>POI:</b> {row.get("category", "unknown")}',
        tooltip=str(row.get('category', 'POI')),
    ).add_to(poi_layer)
poi_layer.add_to(fmap)

# ── LAYER 4 (HIDDEN by default): AADT traffic stations ──────────────────────
# show=False means the checkbox starts UNCHECKED — advanced view only
aadt_show = aadt_with_data[aadt_with_data['latest_aadt'] >= 5000].copy() if len(aadt_with_data) > 0 else aadt_with_data.copy()

aadt_layer = folium.FeatureGroup(
    name=f'AADT Traffic Stations — advanced ({len(aadt_show)} stations)',
    show=False,   # ← HIDDEN BY DEFAULT
)
for _, row in aadt_show.iterrows():
    aadt_val = int(row['latest_aadt'])
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=4 + np.log10(max(aadt_val, 1000)) * 1.5,
        color='#3399FF', weight=1,
        fill=True, fill_color='#66BBFF', fill_opacity=0.4,
        popup=(
            f'<b>TxDOT AADT Station</b><br>'
            f'Station: {row.get("TRFC_STATN_ID", "")}<br>'
            f'Road: {row.get("ON_ROAD", "N/A")}<br>'
            f'AADT: {aadt_val:,} vehicles/day'
        ),
        tooltip=f'AADT: {aadt_val:,}',
    ).add_to(aadt_layer)
aadt_layer.add_to(fmap)

# ── LEGEND ──────────────────────────────────────────────────────────────────
legend_html = '''
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 1000;
     background: white; padding: 12px 16px; border-radius: 8px;
     border: 2px solid #333; font-size: 13px; font-family: sans-serif;
     box-shadow: 0 2px 6px rgba(0,0,0,0.15); max-width: 260px;">
  <b>OOHScout — Real Waco Billboard Market</b><br>
  <i style="font-size: 11px;">Source: live TxDOT ArcGIS API (2026)</i><br><br>

  <b>Real permitted billboards:</b><br>
  <span style="color:#8B0000; font-size: 16px;">●</span> Static bulletin<br>
  <span style="color:#FF2222; font-size: 16px;">●</span> Digital bulletin<br>
  <i style="font-size: 11px;">Click any billboard for permit details + verification links</i><br><br>

  <b>Context:</b><br>
  <span style="color:#CC0066; font-weight: bold;">━━</span> IH-35 highway<br>
  <span style="color:#0066CC">●</span> POI advertiser (fuel, food, hotel, bank)<br><br>

  <b>Advanced (toggle in layer control):</b><br>
  <span style="color:#66BBFF">◯</span> AADT traffic station (size = volume)<br>
</div>
'''
fmap.get_root().html.add_child(folium.Element(legend_html))

# Layer control (Ch4 Step 34 pattern)
folium.LayerControl(collapsed=False, position='topright').add_to(fmap)

# Save the map
map_path = DATA_DIR / 'ch4_waco_billboards_map.html'
fmap.save(str(map_path))

print(f'Map saved: {map_path}')
print(f'File size: {map_path.stat().st_size / 1024:.1f} KB')
print(f'\nMap layers:')
print(f'  ON  — IH-35 highway line')
print(f'  ON  — {len(ih35_bb)} real billboards (with verification links)')
print(f'  ON  — {len(pois)} POI advertisers')
print(f'  OFF — {len(aadt_show)} AADT stations (toggle in layer control to enable)')
print(f'\nOpen this in your browser:')
print(f'  file:///{map_path.resolve().as_posix()}')

# Display in Jupyter (may be blank on complex maps — always trust the saved HTML)
fmap


Map saved: data\new_study\ch4_waco_billboards_map.html
File size: 1430.8 KB

Map layers:
  ON  — IH-35 highway line
  ON  — 114 real billboards (with verification links)
  ON  — 186 POI advertisers
  OFF — 209 AADT stations (toggle in layer control to enable)

Open this in your browser:
  file:///C:/Users/nguye/Documents/billboardAI/notebooks/data/new_study/ch4_waco_billboards_map.html


In [29]:
# ── F.1: Final summary — what this notebook proved ──────────────────────────

print('=' * 70)
print('  OOHScout AI — Chapter 4 Only — Waco Billboard Market')
print('=' * 70)
print()
print('DATA COLLECTED (all from public, verifiable sources):')
print(f'  • {len(billboards):>4} real permitted billboards in McLennan County')
print(f'  • {len(ih35_bb):>4} of those on IH-35 (the corridor we care about)')
print(f'  • {len(pois):>4} advertiser POIs (fuel, food, hotel, bank, etc.)')
print(f'  • {len(ih35_mainlane):>4} IH-35 road segments for context')
print(f'  • {len(aadt_with_data):>4} AADT traffic stations (optional layer)')
print()
print('MARKET INTELLIGENCE (for a Waco operator conversation):')
if len(ih35_bb) > 0:
    op = ih35_bb['OWNR'].value_counts()
    dig = (ih35_bb['ELECTRONIC'] == 'Yes').sum()
    print(f'  Top operator:        {op.index[0]} ({op.iloc[0]} signs, {op.iloc[0]/len(ih35_bb)*100:.0f}% share)')
    print(f'  Unique operators:    {len(op)}')
    print(f'  Digital penetration: {dig}/{len(ih35_bb)} = {dig/len(ih35_bb)*100:.0f}%')
    print(f'  Digital opportunity: {100 - dig/len(ih35_bb)*100:.0f}% of market still static')
print()
print('FILES PRODUCED:')
for f in sorted(DATA_DIR.glob('ch4_*')):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:<45} {size_kb:>8.1f} KB')
print()
print('EVERY TECHNIQUE USED IS FROM CHAPTER 4:')
print('  • ox.features_from_polygon()   — Ch4 Steps 29, 32, 34')
print('  • requests.get() + REST API    — Ch4 Step 34')
print('  • Cache pattern                — Ch4 (every step)')
print('  • .to_crs() metric projection  — Ch4 Steps 27, 29')
print('  • .to_file(driver=GeoJSON)     — Ch4')
print('  • folium.Map + FeatureGroup    — Ch4 Step 34')
print('  • folium.CircleMarker + Popup  — Ch4 Step 34')
print('  • .value_counts() analysis     — Ch4 Step 34')
print()
print('NEXT STEPS (require chapters beyond Ch4):')
print('  Ch6+ → automated candidate ranking (spatial joins)')
print('  Ch7  → detect billboards in aerial imagery (object detection)')
print('  Ch8  → predict revenue from features (spatial regression)')
print('  Ch14 → autonomous agent orchestrating everything (LangGraph)')


  OOHScout AI — Chapter 4 Only — Waco Billboard Market

DATA COLLECTED (all from public, verifiable sources):
  •  182 real permitted billboards in McLennan County
  •  114 of those on IH-35 (the corridor we care about)
  •  186 advertiser POIs (fuel, food, hotel, bank, etc.)
  •  430 IH-35 road segments for context
  •  465 AADT traffic stations (optional layer)

MARKET INTELLIGENCE (for a Waco operator conversation):
  Top operator:        Lamar Advantage Outdoor Company, L.P. (51 signs, 45% share)
  Unique operators:    19
  Digital penetration: 3/114 = 3%
  Digital opportunity: 97% of market still static

FILES PRODUCED:
  ch4_osm_ih35_waco.geojson                        942.5 KB
  ch4_osm_pois_waco.geojson                         26.0 KB
  ch4_txdot_aadt_mclennan.geojson                  459.7 KB
  ch4_txdot_billboards_mclennan.geojson             77.2 KB
  ch4_waco_billboards_map.html                    1430.8 KB

EVERY TECHNIQUE USED IS FROM CHAPTER 4:
  • ox.features_from_polyg